# Binning and Binarization in ML Feature Engineering

# Why binning/discretization?
# Converts continuous numeric features into categorical/discrete intervals ("bins").
# Useful for handling outliers, improving robustness, and sometimes boosting model performance.

import numpy as np
import pandas as pd
from sklearn.preprocessing import KBinsDiscretizer, Binarizer
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# Discretization/Binning strategies:
# 1. Equal Width (Uniform) Binning: All bins have the same width/range.
# 2. Equal Frequency (Quantile) Binning: Each bin contains (approximately) the same number of samples.
# 3. KMeans Binning: Uses clustering to group values into bins.

# Example dataset:
data = pd.read_csv('titanic.csv')
X = data[['Age', 'Fare']]
y = data['Survived']
X['Age'].fillna(X['Age'].mean(), inplace=True)

# 1. Equal Width Binning (strategy='uniform')
width_discretizer = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='uniform')
X_age_width = width_discretizer.fit_transform(X[['Age']])

# 2. Equal Frequency Binning (strategy='quantile')
quantile_discretizer = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile')
X_fare_quantile = quantile_discretizer.fit_transform(X[['Fare']])

# 3. KMeans Binning (strategy='kmeans')
kmeans_discretizer = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='kmeans')
X_age_kmeans = kmeans_discretizer.fit_transform(X[['Age']])

# Use in ColumnTransformer for pipeline workflows
preprocessor = ColumnTransformer([
    ('quantile_age', KBinsDiscretizer(n_bins=3, strategy='quantile', encode='ordinal'), ['Age']),
    ('width_fare', KBinsDiscretizer(n_bins=4, strategy='uniform', encode='ordinal'), ['Fare'])
], remainder='passthrough')

# Fitting and evaluating a model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
model = DecisionTreeClassifier()
model.fit(preprocessor.fit_transform(X_train), y_train)
score = model.score(preprocessor.transform(X_test), y_test)
print(f"Accuracy after binning: {score:.3f}")

# Custom/domain-based binning
# Sometimes, bins are based on business logic (age groups, income brackets)
def custom_age_bin(value):
    if value < 18:
        return 'child'
    elif value < 60:
        return 'adult'
    else:
        return 'senior'
X['AgeGroup'] = X['Age'].apply(custom_age_bin)

# Binarization: Converts numerical features to binary values (0/1) using a threshold
# Example: Income (is_taxable or not), Image pixels (black or white)
binarizer = Binarizer(threshold=6)
X['TravelingAlone'] = binarizer.fit_transform((X['SibSp'] + X['Parch']).values.reshape(-1, 1))

# Practice/Examples:
# Try binning and binarization on real datasets. Compare model accuracy before/after.
# Use each strategy depending on the feature distribution and your domain needs.

# Summary:
# Use KBinsDiscretizer for uniform, quantile, and kmeans binning.
# Use Binarizer for 0/1 conversion; ideal for "yes/no" or "taxable/not" flags.
# Custom bins require manual logic; use for domain-driven feature engineering.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score

from sklearn.compose import ColumnTransformer

In [4]:
df = pd.read_csv('train.csv',usecols=['Age','Fare','SibSp','Parch','Survived'])

In [5]:
df.dropna(inplace = True)

In [6]:
df.head()

,Survived,Age,SibSp,Parch,Fare
0,0,22.0,1,0,7.2500
1,1,38.0,1,0,71.2833
2,1,26.0,0,0,7.9250
3,1,35.0,1,0,53.1000
4,0,35.0,0,0,8.0500


In [7]:
df['family'] = df['SibSp'] + df['Parch']

In [9]:
df.head()

,Survived,Age,SibSp,Parch,Fare,family
0,0,22.0,1,0,7.2500,1
1,1,38.0,1,0,71.2833,1
2,1,26.0,0,0,7.9250,0
3,1,35.0,1,0,53.1000,1
4,0,35.0,0,0,8.0500,0


In [10]:
df.drop(columns=['SibSp','Parch'],inplace = True )

In [11]:
df.head()

,Survived,Age,Fare,family
0,0,22.0,7.2500,1
1,1,38.0,71.2833,1
2,1,26.0,7.9250,0
3,1,35.0,53.1000,1
4,0,35.0,8.0500,0


In [13]:
X = df.drop(columns = ['Survived'])
y= df['Survived']
X

,Age,Fare,family
0,22.0,7.2500,1
1,38.0,71.2833,1
2,26.0,7.9250,0
3,35.0,53.1000,1
4,35.0,8.0500,0
...,...,...,...
885,39.0,29.1250,5
886,27.0,13.0000,0
887,19.0,30.0000,0
889,26.0,30.0000,0


In [15]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2, random_state = 42)

In [26]:
X_train.head()

,Age,Fare,family
328,31.0,20.5250,2
73,26.0,14.4542,1
253,30.0,16.1000,1
719,33.0,7.7750,0
666,25.0,13.0000,0


In [27]:
# without binarization

clf = DecisionTreeClassifier()
clf.fit(X_train,y_train)
y_pred = clf.predict(X_test)
accuracy_score(y_test,y_pred)

0.6293706293706294

In [37]:
np.mean(cross_val_score(clf,X,y,cv=24,scoring='accuracy'))

np.float64(0.6723180076628351)

# Applying Binarization

In [39]:
from sklearn.preprocessing import Binarizer

In [40]:
trf = ColumnTransformer([
    ('bin',Binarizer(copy=False),['family'])
],remainder = 'passthrough')

In [41]:
X_train_trf = trf.fit_transform(X_train)
X_test_trf = trf.transform(X_test)

In [42]:
pd.DataFrame(X_train_trf,columns =['family','Age','Fare'])

,family,Age,Fare
0,1.0,31.0,20.5250
1,1.0,26.0,14.4542
2,1.0,30.0,16.1000
3,0.0,33.0,7.7750
4,0.0,25.0,13.0000
...,...,...,...
566,1.0,46.0,61.1750
567,0.0,25.0,13.0000
568,0.0,41.0,134.5000
569,1.0,33.0,20.5250


In [45]:
clf = DecisionTreeClassifier()
clf.fit(X_train_trf,y_train)
y_pred2 = clf.predict(X_test_trf)
accuracy_score(y_test,y_pred2)

0.5944055944055944

In [46]:
X_trf = trf.fit_transform(X)
np.mean(cross_val_score(DecisionTreeClassifier(),X_trf,y,cv=10,scoring='accuracy'))

np.float64(0.6360328638497652)